# LLM Multi-Turn — Conversation Memory + Token Tracking

Maintain a running **message history** so the model can refer back to earlier turns. After each API call, print **latency** and **token usage** — the same metrics, using each provider’s native SDK.

The conversation covers three turns; the model's final reply should summarize what was discussed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

import time
import os

TURNS = [
    "I'm learning about neural networks. What is a perceptron?",
    'How does that connect to backpropagation?',
    'Summarize everything we just covered in two sentences.'
]

def print_metrics(latency: float, prompt_tok: int, completion_tok: int):
    total = prompt_tok + completion_tok
    print(f'  [latency] {latency:.2f}s  '
          f'[tokens] prompt={prompt_tok}, completion={completion_tok}, total={total}')

---
## Ollama (local)

Same append pattern as OpenAI. Token counts come from `resp['prompt_eval_count']` and `resp['eval_count']`.

In [ ]:
import ollama

OLLAMA_MODEL = 'mistral-nemo:12b-instruct-2407-q4_K_M'

messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    dt = time.time() - t0

    reply = resp['message']['content']
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.get('prompt_eval_count', 0), resp.get('eval_count', 0))
    print()

---
## OpenAI

Build a `messages` list; append each assistant reply before the next user turn. Token counts come from `resp.usage`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = openai_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages
    )
    dt = time.time() - t0

    reply = resp.choices[0].message.content
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print()

---
## Anthropic

Same pattern. Anthropic's `usage` attribute exposes `input_tokens` and `output_tokens`.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL, max_tokens=500, messages=messages
    )
    dt = time.time() - t0

    reply = resp.content[0].text
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.usage.input_tokens, resp.usage.output_tokens)
    print()

---
## Google Gemini

Use `client.chats.create(...)` to maintain history automatically. Token counts come from `resp.usage_metadata`.

In [ ]:
from google import genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)
chat = client.chats.create(model=GOOGLE_MODEL)

for turn in TURNS:
    print(f'User: {turn}')

    t0 = time.time()
    resp = chat.send_message(turn)
    dt = time.time() - t0

    print(f'Assistant: {resp.text}')
    u = resp.usage_metadata
    print_metrics(dt, u.prompt_token_count, u.candidates_token_count)
    print()

---
## 5. Review

The same three-turn conversation ran against four providers, and the mechanics were identical everywhere: chat APIs are **stateless**, so "memory" is nothing more than re-sending the whole message history with every call.

| Provider | History | Prompt tokens | Completion tokens |
|---|---|---|---|
| Ollama | manual `messages` list | `resp['prompt_eval_count']` | `resp['eval_count']` |
| OpenAI | manual `messages` list | `resp.usage.prompt_tokens` | `resp.usage.completion_tokens` |
| Anthropic | manual `messages` list | `resp.usage.input_tokens` | `resp.usage.output_tokens` |
| Google Gemini | `client.chats.create(...)` chat object | `resp.usage_metadata.prompt_token_count` | `resp.usage_metadata.candidates_token_count` |

**Takeaways**

- **Memory is the caller's job.** Ollama, OpenAI, and Anthropic all use the same pattern: append the user turn, call the API, append the assistant reply, repeat. Forgetting to append the assistant reply silently breaks the conversation — the model loses track of its own answers.
- **Gemini's chat object is a convenience, not server-side memory.** `chat.send_message()` does the appending for you inside the SDK; the full history still travels with every request.
- **Prompt tokens grow every turn.** Because the entire history is re-sent, turn 3's prompt contains turns 1 and 2 plus both replies — watch the `prompt=` number climb in the metrics lines. This is why long conversations get progressively slower and more expensive.
- **The last turn is a built-in test.** "Summarize everything we just covered" can only be answered if the earlier turns actually reached the model — a wrong or empty summary means the history wiring is broken.

## Review

**Takeaways**

- **The model is stateless; the conversation is not.** Memory is nothing more than you resending the accumulated message list on every call - there is no hidden server-side thread.
- **That means context grows with every turn**, and so does the prompt-token count. The latency and token figures printed above are the visible cost of a long conversation.
- **Every SDK expresses the same idea differently** - an explicit message list in some, a chat-session object in others - but underneath, all of them are resending history.
- **Track tokens from the start.** Cost and context-limit problems both show up as a slowly growing prompt, and neither is obvious until you are measuring it.